# LeetCode 789 — Escape The Ghosts bằng Python

Link đề: https://leetcode.com/problems/escape-the-ghosts/ · Độ khó: **Medium** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def escapeGhosts(self, ghosts: List[List[int]], target: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bạn xuất phát ở `(0, 0)` trên lưới vô hạn, muốn tới `target` **trước** mọi con ma; mọi bên cùng lúc đi 1 ô (4 hướng) hoặc đứng yên; đến cùng một ô **cùng lúc** với ma thì **không** tính là thoát.
2. Nắm các công cụ Python: **khoảng cách Manhattan** `abs(x1 - x2) + abs(y1 - y2)`, giải nén cặp `x, y = p`, `all(...)`/`any(...)`/`min(...)` với biểu thức sinh, tập hợp `set` để mô phỏng "các ô có thể đang đứng ở lượt `t`", và so sánh **chặt** (`<`) với **không chặt** (`<=`).
3. Đi từ **mô phỏng trò chơi theo từng lượt (BFS theo lớp thời gian)** → **công thức khoảng cách** → **kiểm tra từng ô trên đường đi thẳng của mình** → **mô phỏng "mọi ma lao thẳng về đích"**.
4. Hiểu mấu chốt: ma giỏi nhất **không cần đón đầu bạn trên đường**; chỉ cần **tới đích trước hoặc cùng lúc** rồi đứng đợi. Vì vậy bạn thoát khi và chỉ khi **khoảng cách của bạn tới đích nhỏ hơn hẳn khoảng cách của mọi ma tới đích**.
5. Biết **cách kiểm chứng một công thức toán học** bằng một **oracle độc lập** (mô phỏng trò chơi), và tránh các lỗi kinh điển: dùng `>=` thay `>`, đo khoảng cách ma–bạn thay vì ma–đích, `return True` sớm trong vòng lặp, khoảng cách Euclid, chỉ xét ma "nằm giữa đường".

## 2. Đề bài & ví dụ

Bạn chơi một trò Pac-Man đơn giản trên lưới 2 chiều **vô hạn**. Bạn bắt đầu ở `[0, 0]` và muốn đến điểm `target = [x_target, y_target]`. Có nhiều con ma, vị trí xuất phát cho bởi mảng `ghosts`, `ghosts[i] = [x_i, y_i]`. Mọi tọa độ là **số nguyên**.

Mỗi lượt, **bạn và mọi con ma** có thể **độc lập** chọn: đi 1 đơn vị theo một trong bốn hướng (Bắc, Đông, Nam, Tây) hoặc **đứng yên**. Mọi hành động diễn ra **đồng thời**.

Bạn **thoát** khi và chỉ khi bạn **đến `target` trước** khi bất kỳ con ma nào đến được chỗ bạn. Nếu bạn và một con ma đến **cùng một ô** (kể cả `target`) **cùng lúc**, **không** tính là thoát.

Trả về `True` nếu bạn **có thể thoát bất kể ma đi thế nào**, ngược lại `False`.

**Ví dụ 1:** `ghosts = [[1,0],[0,3]]`, `target = [0,1]` → `True` (bạn tới `(0,1)` sau 1 lượt, hai ma không kịp).

**Ví dụ 2:** `ghosts = [[1,0]]`, `target = [2,0]` → `False` (con ma ở `(1,0)` nằm giữa bạn và đích).

**Ví dụ 3:** `ghosts = [[2,0]]`, `target = [1,0]` → `False` (ma tới đích **cùng lúc** với bạn).

**Ràng buộc:** `1 <= ghosts.length <= 100`; `ghosts[i].length == 2`; `-10^4 <= x_i, y_i <= 10^4`; có thể **nhiều ma cùng vị trí**; `target.length == 2`; `-10^4 <= x_target, y_target <= 10^4`.

In [1]:
import itertools
import random
import time
from typing import List

EX = [
    ([[1, 0], [0, 3]], [0, 1], True),
    ([[1, 0]], [2, 0], False),
    ([[2, 0]], [1, 0], False),
]
for ghosts, target, want in EX:
    print(ghosts, target, "->", want)

[[1, 0], [0, 3]] [0, 1] -> True
[[1, 0]] [2, 0] -> False
[[2, 0]] [1, 0] -> False


## 3. Kiến thức nền

### 3.1. Khoảng cách Manhattan

Đi 4 hướng trên lưới, mỗi bước đổi **một** tọa độ đúng 1 đơn vị. Số bước **tối thiểu** từ `(x1, y1)` đến `(x2, y2)` là `|x1 − x2| + |y1 − y2|` — gọi là khoảng cách **Manhattan**. `abs(...)` là giá trị tuyệt đối. Ví dụ từ `(0,0)` đến `(3,−2)` cần `3 + 2 = 5` bước (không thể ít hơn, vì mỗi bước chỉ thay đổi một tọa độ 1 đơn vị).

In [2]:
def dist(p, q):
    return abs(p[0] - q[0]) + abs(p[1] - q[1])


print(dist((0, 0), (3, -2)), dist((1, 0), (2, 0)), dist((5, 5), (5, 5)))
x, y = (3, -2)                                 # giải nén cặp
print(x, y)

5 1 0
3 -2


### 3.2. `all`, `any`, `min` với biểu thức sinh

- `all(cond for g in ghosts)`: `True` nếu **mọi** phần tử thỏa; **dừng sớm** khi gặp một phần tử không thỏa.
- `any(cond for g in ghosts)`: `True` nếu **có ít nhất một** phần tử thỏa.
- `min(f(g) for g in ghosts)`: giá trị nhỏ nhất của `f`.

Dãy rỗng: `all([]) = True`, `any([]) = False`. (Đề đảm bảo có ít nhất 1 ma.)

In [3]:
ghosts = [[1, 0], [0, 3], [5, 5]]
target = [0, 1]
d = [dist(g, target) for g in ghosts]
print(d, min(d))
print(all(x > 1 for x in d), any(x <= 1 for x in d), all(x > 1 for x in []), any(x > 1 for x in []))

[2, 2, 9] 2
True False True False


### 3.3. So sánh chặt và không chặt

"Đến **trước**" nghĩa là **nhỏ hơn hẳn** (`<`), còn "đến cùng lúc **không** tính" nghĩa là dấu bằng thuộc về **ma** (ma thắng khi `d_ma <= d_bạn`). Nhầm `<` với `<=` là lỗi số một của bài này; hãy thử với số nhỏ: bạn cần 1 lượt, ma cần 1 lượt ⇒ hòa ⇒ **không** thoát.

In [4]:
d_me, d_ghost = 1, 1
print("bạn thoát nếu d_me < d_ghost:", d_me < d_ghost, "| nếu viết nhầm d_me <= d_ghost:", d_me <= d_ghost)

bạn thoát nếu d_me < d_ghost: False | nếu viết nhầm d_me <= d_ghost: True


### 3.4. "Quả bóng" các ô ma có thể tới: hình thoi Manhattan

Ma có thể **đứng yên** nên sau `t` lượt nó có thể đang ở **bất kỳ** ô nào cách nó **không quá `t`** (khoảng cách Manhattan `≤ t`): một **hình thoi** (kim cương) bán kính `t`. Bạn an toàn ở lượt `t` tại ô `p` khi **không** con ma nào có `dist(ghost, p) <= t`. Đây là cơ sở của mô phỏng ở Cách 1.

In [5]:
def ball(center, t):
    cx, cy = center
    return {(cx + dx, cy + dy) for dx in range(-t, t + 1) for dy in range(-t, t + 1) if abs(dx) + abs(dy) <= t}


for t in range(4):
    print(f"bán kính {t}: {len(ball((0, 0), t))} ô")           # 1, 5, 13, 25 = 2t^2 + 2t + 1
assert all(len(ball((0, 0), t)) == 2 * t * t + 2 * t + 1 for t in range(10))

bán kính 0: 1 ô
bán kính 1: 5 ô
bán kính 2: 13 ô
bán kính 3: 25 ô


### 3.5. Bất đẳng thức tam giác (khoảng cách Manhattan)

Với ba điểm `A, B, C`: `dist(A, C) ≤ dist(A, B) + dist(B, C)`. Hệ quả dùng ở mục 10: nếu một con ma tới được một ô `P` trên đường của bạn **không muộn hơn** bạn, thì nó tới được **đích** cũng **không muộn hơn** bạn.

In [6]:
rng = random.Random(1)
for _ in range(10000):
    A, B, C = [(rng.randint(-9, 9), rng.randint(-9, 9)) for _ in range(3)]
    assert dist(A, C) <= dist(A, B) + dist(B, C)
print("bất đẳng thức tam giác: OK trên 10000 bộ ba ngẫu nhiên")

bất đẳng thức tam giác: OK trên 10000 bộ ba ngẫu nhiên


### 3.6. Tập hợp `set` theo lớp thời gian

Để xét mọi ô bạn có thể đứng sau `t` lượt, ta dùng **tập** `S_t`; `S_{t+1}` gồm các ô cách một ô của `S_t` (hoặc chính nó, vì được đứng yên) mà **an toàn**. Toán tử: `{... for p in S for dx, dy in MOVES}`, `target in S`, `not S` (rỗng).

In [7]:
MOVES = [(0, 0), (1, 0), (-1, 0), (0, 1), (0, -1)]            # đứng yên + 4 hướng
S = {(0, 0)}
S = {(x + dx, y + dy) for x, y in S for dx, dy in MOVES}
print(sorted(S))
S = {(x + dx, y + dy) for x, y in S for dx, dy in MOVES}
print(len(S), (1, 1) in S, (3, 0) in S, bool(S), not set())

[(-1, 0), (0, -1), (0, 0), (0, 1), (1, 0)]
13 True False True True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy xem **Ví dụ 2**: bạn ở `(0,0)`, đích `(2,0)`, ma ở `(1,0)`.

- Bạn cần **2 lượt** đi thẳng (qua `(1,0)` rồi tới `(2,0)`).
- Ma ở `(1,0)` chỉ cần **1 lượt** để tới `(2,0)`. Nó đến đích **trước** bạn rồi đứng đợi; khi bạn tới, bạn **chạm ma** ⇒ thua.

**Ví dụ 3:** đích `(1,0)`, ma ở `(2,0)`: bạn mất 1 lượt; ma cũng 1 lượt ⇒ **cùng lúc** ⇒ không tính thoát.

**Ví dụ 1:** đích `(0,1)`: bạn mất 1 lượt. Ma `(1,0)` cách đích `|1−0| + |0−1| = 2` lượt; ma `(0,3)` cách `2` lượt. Cả hai `> 1` ⇒ bạn tới trước ⇒ thoát.

**Câu hỏi then chốt:** ma có cần "chặn đường" bạn không? Không! Nếu bạn muốn tới đích, ma **chỉ cần đứng ở đích đợi**. Cách ma tốt nhất là **tới đích càng sớm càng tốt**. Nếu như vậy vẫn không kịp (ma cách đích **xa hơn** bạn), thì nó cũng không thể chặn bạn ở một ô khác trên đường: sẽ chứng minh ở mục 10.

Vậy bài toán trò chơi "đối kháng" thu về một phép so sánh số: **`d_bạn < d_ma` cho mọi ma**, với `d_bạn = |x_t| + |y_t|` và `d_ma = |x − x_t| + |y − y_t|`.

## 5. Cách 1 — Brute Force / baseline: mô phỏng trò chơi theo **từng lượt** (BFS theo lớp thời gian)

**Ý tưởng:** dịch đề thành thuật toán. Tại mỗi lượt `t`, ta duy trì tập `S_t` = **các ô bạn có thể đang đứng an toàn**:

- `S_0 = {(0,0)}` nếu không có ma ở `(0,0)` (ngược lại thua ngay).
- `S_{t+1}` = các ô cách một ô của `S_t` (kể cả đứng yên) **trong một cửa sổ hữu hạn quanh các điểm**, và **không** thuộc "quả bóng" của **bất kỳ ma nào** ở lượt `t+1` (tức `dist(ghost, ô) > t + 1` với mọi ma).
- Nếu `target ∈ S_t` ở lượt nào đó: thoát (`True`). Nếu `S_t` rỗng: không thoát (`False`).

Ý nghĩa: ô `p` ở lượt `t` bị loại nếu **có một con ma** có thể đang đứng ở đó (`dist(g, p) ≤ t`), nghĩa là có cách ma đi để chạm bạn. Một đường đi của bạn "an toàn bất kể ma đi thế nào" khi **mọi ô của nó** ở đúng thời điểm đều nằm ngoài các quả bóng.

**Cửa sổ hữu hạn:** ta chỉ cho phép bạn đi trong hình chữ nhật bao các điểm (gốc, đích, ma) cộng lề; đủ để tìm đường vòng nếu có. Đây là **oracle độc lập** để kiểm tra công thức ở mục sau.

In [8]:
MOVES4 = [(0, 0), (1, 0), (-1, 0), (0, 1), (0, -1)]


def escape_bfs(ghosts, target, delay=0, moves=MOVES4, metric=dist, margin=3):
    """Oracle: tồn tại đường đi của mình mà mọi ô ở mọi thời điểm đều ngoài 'quả bóng' của mọi ma?"""
    xs = [0, target[0]] + [g[0] for g in ghosts]
    ys = [0, target[1]] + [g[1] for g in ghosts]
    lo_x, hi_x = min(xs) - margin, max(xs) + margin
    lo_y, hi_y = min(ys) - margin, max(ys) + margin

    def unsafe(p, t):
        return any(metric(g, p) <= t for g in ghosts)

    if unsafe((0, 0), 0):
        return False
    for t in range(1, delay + 1):                     # (tùy chọn) buộc đứng yên ở gốc `delay` lượt đầu
        if unsafe((0, 0), t):
            return False
    S = {(0, 0)}
    t = delay
    while S:
        if tuple(target) in S:
            return True
        t += 1
        S = {(x + dx, y + dy) for x, y in S for dx, dy in moves
             if lo_x <= x + dx <= hi_x and lo_y <= y + dy <= hi_y and not unsafe((x + dx, y + dy), t)}
    return False


for ghosts, target, want in EX:
    got = escape_bfs(ghosts, target)
    print(got, want)
    assert got == want

True True
False False
False False


**Độ phức tạp:** gọi `W` là số ô trong cửa sổ, `T` số lượt tối đa, `g` số ma: mỗi lượt xử lý tối đa `W` ô × 5 hướng × `g` phép kiểm tra ⇒ `O(T · W · g)`. Với tọa độ tới `10^4`, `W ≈ 4·10^8` và `T ≈ 4·10^4` — **không khả thi**; chỉ dùng cho input nhỏ làm oracle.

**Nút thắt:** ta mô phỏng **cả cuộc chơi** trong khi câu trả lời dường như chỉ phụ thuộc vào **vài con số khoảng cách**. Câu hỏi: có thể thay "mô phỏng đối kháng" bằng một phép so sánh không?

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề 1 — ma chỉ cần chạy tới đích.** Nếu có ma `g` thỏa `dist(g, target) ≤ dist((0,0), target)`, thì bạn **không** thể thoát: ma đi thẳng tới `target` (mất `dist(g, target)` lượt) rồi **đứng đợi**; khi bạn tới `target` (không sớm hơn `d_bạn` lượt) ma **đã ở đó** hoặc tới cùng lúc ⇒ không thoát. Mà bạn **bắt buộc** phải tới `target`.

**Bổ đề 2 — nếu mọi ma ở xa hơn thì bạn thoát.** Giả sử `dist(g, target) > d_bạn` với mọi ma. Bạn đi **thẳng tới đích** theo đường ngắn nhất (`d_bạn` lượt). Giả sử ma `g` chạm bạn ở ô `P` tại lượt `t` (nghĩa là `dist(g, P) ≤ t`). Bạn đã đi `t` bước tới `P` trên đường ngắn nhất nên `dist(P, target) = d_bạn − t`. Theo bất đẳng thức tam giác: `dist(g, target) ≤ dist(g, P) + dist(P, target) ≤ t + (d_bạn − t) = d_bạn`. Mâu thuẫn với `dist(g, target) > d_bạn`. Vậy không ma nào chạm bạn.

**Kết luận:**

```
escape  ⇔  dist((0,0), target) < dist(g, target)   với mọi ma g
```

Từ bổ đề này ta có các cách: **công thức khoảng cách** (Cách 2), **kiểm tra từng ô trên đường đi thẳng** (Cách 3, đúng theo chứng minh bổ đề 2) và **mô phỏng "mọi ma lao thẳng về đích"** (Cách 4, đúng theo bổ đề 1).

## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (mô phỏng, mục 5) và ba cách dưới đây. Cách 2 là **phép so sánh số** (ý tưởng cuối cùng); Cách 3 **kiểm tra trực tiếp một đường đi cụ thể** của bạn (không cần suy luận về đích); Cách 4 **mô phỏng chiến lược của ma** (mọi ma chạy tới đích).

### Cách 2 — So sánh khoảng cách tới đích (cách chính thức)

**Giải thích từng đoạn:**

1. `d = |x_t| + |y_t|`: số lượt bạn cần (từ gốc).
2. Với mỗi ma `(x, y)`: `|x − x_t| + |y − y_t|` là số lượt ma cần để tới đích.
3. Trả về `True` khi **mọi** ma đều `> d` (chặt). Chỉ cần **một** ma `≤ d` là `False`.

In [9]:
def escape_formula(ghosts, target):
    tx, ty = target
    d = abs(tx) + abs(ty)
    return all(abs(x - tx) + abs(y - ty) > d for x, y in ghosts)


for ghosts, target, want in EX:
    got = escape_formula(ghosts, target)
    print(got, want)
    assert got == want

True True
False False
False False


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def escapeGhosts(self, ghosts: List[List[int]], target: List[int]) -> bool:
        tx, ty = target
        d = abs(tx) + abs(ty)
        return all(abs(x - tx) + abs(y - ty) > d for x, y in ghosts)
```

**Độ phức tạp:** một lượt qua `g` ma ⇒ **`O(g)`** thời gian, `O(1)` bộ nhớ.

### Cách 3 — Kiểm tra từng ô trên **đường đi thẳng** của bạn

**Giải thích từng đoạn:**

1. Chọn một đường đi cố định của bạn: đi theo trục `x` tới `x_t`, rồi theo trục `y` tới `y_t`; ô ở lượt `t` là `P_t` (`t = 0..d`).
2. Một con ma **chặn được** bạn tại `P_t` khi `dist(ghost, P_t) ≤ t` (kịp tới `P_t` vào lượt `t` hoặc sớm hơn rồi đợi).
3. Nếu **bất kỳ** ô nào bị chặn thì `False`; nếu cả đường đi sạch thì `True`.

Cách này **không dùng** suy luận "ma chạy tới đích" — nó chỉ kiểm tra một đường cụ thể, và bổ đề 2 chứng minh rằng đường thẳng là **đường tốt nhất** (nếu đường thẳng bị chặn thì ma tới đích kịp).

In [10]:
def escape_path(ghosts, target):
    tx, ty = target
    # các ô bạn đi qua: lượt t (0..d)
    path = [(0, 0)]
    x = y = 0
    while x != tx:
        x += 1 if tx > x else -1
        path.append((x, y))
    while y != ty:
        y += 1 if ty > y else -1
        path.append((x, y))
    for t, p in enumerate(path):
        for g in ghosts:
            if dist(g, p) <= t:                      # ma kịp tới p vào lượt t (hoặc sớm hơn)
                return False
    return True


for ghosts, target, want in EX:
    got = escape_path(ghosts, target)
    print(got, want)
    assert got == want

True True
False False
False False


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def escapeGhosts(self, ghosts: List[List[int]], target: List[int]) -> bool:
        tx, ty = target
        x = y = 0
        pts = [(0, 0)]
        while x != tx:
            x += 1 if tx > x else -1
            pts.append((x, y))
        while y != ty:
            y += 1 if ty > y else -1
            pts.append((x, y))
        return not any(abs(gx - px) + abs(gy - py) <= t
                       for t, (px, py) in enumerate(pts)
                       for gx, gy in ghosts)
```

**Độ phức tạp:** đường đi có `d + 1 ≤ 2·10^4 + 1` ô, mỗi ô kiểm tra `g ≤ 100` ma ⇒ `O(d · g)` ≈ `2·10^6` phép; `O(d)` bộ nhớ cho danh sách ô (có thể `O(1)` nếu sinh từng ô).

### Cách 4 — Mô phỏng song song: bạn **và mọi ma** cùng lao thẳng về đích

**Giải thích từng đoạn:**

1. Ma thông minh nhất làm gì? Chạy **thẳng tới đích** rồi đứng đợi (bổ đề 1). Ta mô phỏng đúng điều đó: mỗi lượt, bạn **và** mọi ma tiến 1 bước về đích (đi theo `x` trước rồi `y`; ma ở đích thì đứng yên).
2. Sau mỗi lượt nếu có ma **trùng vị trí** với bạn thì `False`; nếu bạn tới đích mà chưa lần nào trùng thì `True`. Trước lượt đầu: nếu có ma ở `(0,0)` thì `False`.

Khác Cách 3 (kiểm tra đường đi bằng bất đẳng thức), ở đây ta **thực sự di chuyển** mọi ma.

In [11]:
def step_toward(p, target):
    x, y = p
    tx, ty = target
    if x != tx:
        x += 1 if tx > x else -1
    elif y != ty:
        y += 1 if ty > y else -1
    return [x, y]


def escape_sim(ghosts, target):
    me = [0, 0]
    gs = [list(g) for g in ghosts]
    if any(g == me for g in gs):
        return False
    while me != list(target):
        me = step_toward(me, target)
        gs = [step_toward(g, target) for g in gs]
        if any(g == me for g in gs):
            return False
    return True


for ghosts, target, want in EX:
    got = escape_sim(ghosts, target)
    print(got, want)
    assert got == want

True True
False False
False False


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def escapeGhosts(self, ghosts: List[List[int]], target: List[int]) -> bool:
        def step(p):
            x, y = p
            if x != target[0]:
                x += 1 if target[0] > x else -1
            elif y != target[1]:
                y += 1 if target[1] > y else -1
            return [x, y]

        me = [0, 0]
        gs = [list(g) for g in ghosts]
        if me in gs:
            return False
        while me != target:
            me = step(me)
            gs = [step(g) for g in gs]
            if me in gs:
                return False
        return True
```

**Độ phức tạp:** tối đa `d ≤ 2·10^4` lượt, mỗi lượt di chuyển `g ≤ 100` ma và so sánh ⇒ `O(d · g)` ≈ `2·10^6`–`4·10^6` phép; `O(g)` bộ nhớ.

### Kiểm tra nhanh bốn cách trên ba ví dụ và hai bẫy

Bẫy 1: ma **phía sau** bạn nhưng vẫn kịp tới đích. Bẫy 2: ma đứng đúng **đích**.

In [12]:
IMPLS = {
    "bfs": escape_bfs,
    "formula": escape_formula,
    "path": escape_path,
    "sim": escape_sim,
}
traps = [([[-1, 0]], [1, 0], True),            # ma phía sau, cách đích 2 > 1: bạn thoát
         ([[0, 5], [3, 3]], [3, 3], False),    # ma đứng đúng đích
         ([[1, 1]], [1, 0], False)]            # ma không nằm "giữa đường" nhưng tới đích cùng lúc
for ghosts, target, want in EX + traps:
    row = {name: f(ghosts, target) for name, f in IMPLS.items()}
    print(ghosts, target, row, want)
    assert all(v == want for v in row.values())

[[1, 0], [0, 3]] [0, 1] {'bfs': True, 'formula': True, 'path': True, 'sim': True} True
[[1, 0]] [2, 0] {'bfs': False, 'formula': False, 'path': False, 'sim': False} False
[[2, 0]] [1, 0] {'bfs': False, 'formula': False, 'path': False, 'sim': False} False
[[-1, 0]] [1, 0] {'bfs': True, 'formula': True, 'path': True, 'sim': True} True
[[0, 5], [3, 3]] [3, 3] {'bfs': False, 'formula': False, 'path': False, 'sim': False} False
[[1, 1]] [1, 0] {'bfs': False, 'formula': False, 'path': False, 'sim': False} False


## 8. TRACE từng bước

**Trace Cách 2 (công thức) trên ba ví dụ:**

| Ví dụ | `target` | `d` (bạn) | Khoảng cách từng ma tới đích | `> d` với mọi ma? | Kết quả |
|---|---|---|---|---|---|
| 1 | `(0,1)` | 1 | ma `(1,0)`: 2; ma `(0,3)`: 2 | 2 > 1, 2 > 1 ✓ | `True` |
| 2 | `(2,0)` | 2 | ma `(1,0)`: 1 | 1 > 2? ✗ | `False` |
| 3 | `(1,0)` | 1 | ma `(2,0)`: 1 | 1 > 1? ✗ | `False` |

In [13]:
def trace_formula(ghosts, target):
    tx, ty = target
    d = abs(tx) + abs(ty)
    print(f"target={target}: bạn cần d = |{tx}| + |{ty}| = {d} lượt")
    ok = True
    for g in ghosts:
        dg = abs(g[0] - tx) + abs(g[1] - ty)
        good = dg > d
        ok &= good
        print(f"  ma {g}: |{g[0]}-({tx})| + |{g[1]}-({ty})| = {dg}  ->  {dg} > {d}? {good}")
    print("  =>", ok, "\n")
    return ok


for ghosts, target, want in EX:
    assert trace_formula(ghosts, target) == want

target=[0, 1]: bạn cần d = |0| + |1| = 1 lượt
  ma [1, 0]: |1-(0)| + |0-(1)| = 2  ->  2 > 1? True
  ma [0, 3]: |0-(0)| + |3-(1)| = 2  ->  2 > 1? True
  => True 

target=[2, 0]: bạn cần d = |2| + |0| = 2 lượt
  ma [1, 0]: |1-(2)| + |0-(0)| = 1  ->  1 > 2? False
  => False 

target=[1, 0]: bạn cần d = |1| + |0| = 1 lượt
  ma [2, 0]: |2-(1)| + |0-(0)| = 1  ->  1 > 1? False
  => False 



**Trace Cách 1 (BFS theo lớp thời gian) trên Ví dụ 2:** in `S_t` và các ô bị loại.

In [14]:
def trace_bfs(ghosts, target, margin=1):
    xs = [0, target[0]] + [g[0] for g in ghosts]
    ys = [0, target[1]] + [g[1] for g in ghosts]
    lo_x, hi_x, lo_y, hi_y = min(xs) - margin, max(xs) + margin, min(ys) - margin, max(ys) + margin
    print(f"cửa sổ x∈[{lo_x},{hi_x}], y∈[{lo_y},{hi_y}], ma {ghosts}, đích {target}")
    S = {(0, 0)}
    t = 0
    print(f"t=0: S={sorted(S)}")
    while S:
        if tuple(target) in S:
            print("  đích ∈ S -> THOÁT")
            return True
        t += 1
        cand = {(x + dx, y + dy) for x, y in S for dx, dy in MOVES4
                if lo_x <= x + dx <= hi_x and lo_y <= y + dy <= hi_y}
        dead = {p for p in cand if any(dist(g, p) <= t for g in ghosts)}
        S = cand - dead
        print(f"t={t}: ứng viên {len(cand)} ô, bị ma chặn {sorted(dead)}, còn S={sorted(S)}")
    print("  S rỗng -> KHÔNG thoát")
    return False


assert trace_bfs([[1, 0]], [2, 0]) is False
print()
assert trace_bfs([[0, 3]], [0, 1]) is True

cửa sổ x∈[-1,3], y∈[-1,1], ma [[1, 0]], đích [2, 0]
t=0: S=[(0, 0)]
t=1: ứng viên 5 ô, bị ma chặn [(0, 0), (1, 0)], còn S=[(-1, 0), (0, -1), (0, 1)]
t=2: ứng viên 8 ô, bị ma chặn [(-1, 0), (0, -1), (0, 0), (0, 1), (1, -1), (1, 1)], còn S=[(-1, -1), (-1, 1)]
t=3: ứng viên 5 ô, bị ma chặn [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1)], còn S=[]
  S rỗng -> KHÔNG thoát

cửa sổ x∈[-1,1], y∈[-1,4], ma [[0, 3]], đích [0, 1]
t=0: S=[(0, 0)]
t=1: ứng viên 5 ô, bị ma chặn [], còn S=[(-1, 0), (0, -1), (0, 0), (0, 1), (1, 0)]
  đích ∈ S -> THOÁT


**Trace Cách 4 (mô phỏng song song) trên Ví dụ 3** (`ma (2,0)`, đích `(1,0)`): in vị trí của bạn và của ma mỗi lượt.

In [15]:
def trace_sim(ghosts, target):
    me = [0, 0]
    gs = [list(g) for g in ghosts]
    print(f"t=0: bạn {me}, ma {gs}")
    t = 0
    if any(g == me for g in gs):
        print("  ma ở ngay vị trí xuất phát -> THUA")
        return False
    while me != list(target):
        t += 1
        me = step_toward(me, target)
        gs = [step_toward(g, target) for g in gs]
        hit = any(g == me for g in gs)
        print(f"t={t}: bạn {me}, ma {gs}{'  <- TRÙNG VỊ TRÍ' if hit else ''}")
        if hit:
            return False
    print("  bạn tới đích an toàn")
    return True


assert trace_sim([[2, 0]], [1, 0]) is False
assert trace_sim([[1, 4], [0, 5]], [0, 2]) is True

t=0: bạn [0, 0], ma [[2, 0]]
t=1: bạn [1, 0], ma [[1, 0]]  <- TRÙNG VỊ TRÍ
t=0: bạn [0, 0], ma [[1, 4], [0, 5]]
t=1: bạn [0, 1], ma [[0, 4], [0, 4]]
t=2: bạn [0, 2], ma [[0, 3], [0, 3]]
  bạn tới đích an toàn


## 9. Code nộp LeetCode

Ta nộp **Cách 2** (công thức khoảng cách): một dòng chính, `O(g)`, dễ chứng minh.

In [16]:
class Solution:
    def escapeGhosts(self, ghosts: List[List[int]], target: List[int]) -> bool:
        tx, ty = target
        d = abs(tx) + abs(ty)
        return all(abs(x - tx) + abs(y - ty) > d for x, y in ghosts)


for ghosts, target, want in EX:
    assert Solution().escapeGhosts(ghosts, target) == want
print([Solution().escapeGhosts(g, t) for g, t, _ in EX])
IMPLS["Solution"] = lambda g, t: Solution().escapeGhosts(g, t)

[True, False, False]


## 10. Vì sao đúng

Gọi `D = dist((0,0), target)` và `Dg = dist(g, target)`.

**(⇒) Nếu có ma `g` với `Dg ≤ D` thì bạn không thể thoát.** Ma đi thẳng tới `target` (mất `Dg ≤ D` lượt) rồi **đứng yên**. Bạn **buộc phải** tới `target`, và để làm vậy cần **ít nhất `D` lượt** (không thể nhanh hơn khoảng cách Manhattan). Tại thời điểm bạn tới (lượt `≥ D`), ma đã ở `target` từ lượt `Dg ≤ D` trở đi: bạn và ma ở **cùng ô cùng lúc** (hoặc ma tới trước) ⇒ không tính là thoát. Chiến lược này của ma **không phụ thuộc** vào cách bạn đi, nên đúng với mọi chiến lược của bạn.

**(⇐) Nếu mọi ma có `Dg > D` thì bạn thoát.** Bạn đi **theo đường ngắn nhất** tới `target` (`D` lượt). Giả sử một ma `g` có thể chạm bạn ở ô `P` tại lượt `t` (`0 ≤ t ≤ D`): ma phải tới `P` không muộn hơn lượt `t`, tức `dist(g, P) ≤ t`. Vì bạn đi đường ngắn nhất nên `dist(P, target) = D − t`. Bất đẳng thức tam giác cho `Dg ≤ dist(g, P) + dist(P, target) ≤ t + (D − t) = D`, mâu thuẫn với `Dg > D`. Vậy không ma nào chạm bạn trên cả đường đi, kể cả ở `target` (`t = D`).

Hai chiều cho: `escape ⇔ D < Dg` với mọi ma `g`. Đó là công thức của Cách 2.

- **Cách 3** là bản "kiểm tra thực tế" của chiều (⇐): nó kiểm tra đúng điều kiện `dist(g, P_t) ≤ t` trên đường thẳng. Nếu sai tại một ô thì chiều (⇐) cho `Dg ≤ D`, nên chiều (⇒) cho `False` — hai bên trùng nhau.
- **Cách 4** là bản "thực hiện chiến lược ma" của chiều (⇒) cộng với tính chất "ma lao thẳng tới đích trên đường cách đích `Dg − t`; nó chỉ có thể trùng với bạn (cách đích `D − t`) khi `Dg ≤ D`".
- **Cách 1** định nghĩa thoát theo "mọi ô trên đường đi nằm ngoài quả bóng của mọi ma ở đúng thời điểm" — một oracle không dùng công thức, nên là phép thử độc lập cho cả (⇒) lẫn (⇐).

Chú ý trường hợp biên: ma **ở gốc** `(0,0)` (`dist(g, (0,0)) = 0 ≤ 0`): khi đó `Dg = dist((0,0), target) = D` ⇒ `Dg ≤ D` ⇒ công thức cho `False` ✓. Ma đứng **trên đích**: `Dg = 0 ≤ D` ⇒ `False` ✓ (kể cả `target = (0,0)`: `D = 0`, `Dg > 0` chỉ khi ma không ở gốc; ràng buộc đề không cấm `target = (0,0)`).

## 11. Phân tích Time / Space Complexity

Gọi `g` là số ma (`≤ 100`), `D = |x_t| + |y_t| ≤ 2·10^4`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS theo lớp thời gian | `O(T · W · g)` | `O(W)` | chỉ làm oracle trên input nhỏ |
| 2. Công thức khoảng cách | `O(g)` | `O(1)` | **nộp bài** |
| 3. Kiểm tra từng ô trên đường thẳng | `O(D · g)` | `O(D)` (hoặc `O(1)`) | trực quan, kiểm tra một đường cụ thể |
| 4. Mô phỏng song song | `O(D · g)` | `O(g)` | thực hiện chiến lược của ma |

Với `g ≤ 100`, `D ≤ 2·10^4` cả Cách 2–4 đều chạy nhanh (Cách 3, 4 cỡ vài triệu phép); chỉ Cách 1 không dùng được ở kích thước đề.

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** (3 ví dụ) cho 4 cách + `Solution`.
2. **Edge case:** một ma ngay gốc, ma trên đích, đích ở gốc, ma phía sau, nhiều ma cùng vị trí, tọa độ cực biên `±10^4`, `100` ma.
3. **Vét cạn:** mọi cấu hình một ma trong `[-3, 3]²` và đích trong `[-3, 3]²`; mọi cặp hai ma và đích trong `[-2, 2]²` — oracle BFS đối chiếu công thức.
4. **Ngẫu nhiên có seed:** `1..4` ma, tọa độ `[-6, 6]`, oracle BFS; có cả đáp án `True` và `False`.
5. **Tọa độ lớn:** `100` ma, tọa độ tới `10^4`: so Cách 2, 3, 4 (Cách 1 quá chậm).
6. **Tính chất:** thêm ma chỉ làm đáp án xấu đi (đơn điệu), ma đặt trên đích ⇒ `False`, dời mọi điểm theo cùng một vector không áp dụng (gốc cố định) nhưng **phản xạ trục** thì áp dụng.
7. **Thời gian.**

In [17]:
def check_all(ghosts, target, expected=None, with_bfs=True):
    ref = escape_formula(ghosts, target)
    if expected is not None:
        assert ref == expected, (ghosts, target, ref, expected)
    for name, f in IMPLS.items():
        if name == "bfs" and not with_bfs:
            continue
        assert f(ghosts, target) == ref, (name, ghosts, target)
    return ref


# 1) Ví dụ đề
for ghosts, target, want in EX:
    check_all(ghosts, target, want)
    assert Solution().escapeGhosts(ghosts, target) == want
print("Ví dụ đề: OK")

# 2) Edge case
check_all([[0, 0]], [3, 3], False)                          # ma ở gốc
check_all([[3, 3]], [3, 3], False)                          # ma trên đích
check_all([[5, 5]], [0, 0], True)                           # đích ở gốc, ma ở xa: bạn đã ở đích
check_all([[0, 0]], [0, 0], False)                          # đích ở gốc và ma cũng ở gốc
check_all([[-1, 0]], [1, 0], True)                          # ma phía sau
check_all([[-3, 0]], [1, 0], True)
check_all([[2, 0], [2, 0], [2, 0]], [1, 0], False)          # nhiều ma cùng vị trí
check_all([[9, 9], [9, 9]], [1, 1], True)
check_all([[1, 1]], [0, 1], False)                          # tới đích cùng lúc (1 lượt) dù không nằm trên đường
check_all([[-10**4, 10**4]] * 100, [10**4, 10**4], False, with_bfs=False)           # ma và bạn cùng cách đích 2*10^4: hòa
check_all([[-10**4, 10**4]] * 100, [10**4, 10**4 - 1], True, with_bfs=False)         # bạn gần hơn 2 lượt
print("Edge case: OK")

Ví dụ đề: OK


Edge case: OK


In [18]:
# 3) Vét cạn
cnt = {True: 0, False: 0}
R = range(-3, 4)
for g in itertools.product(R, R):
    for t in itertools.product(R, R):
        cnt[check_all([list(g)], list(t))] += 1
print("một ma trong [-3,3]^2, đích trong [-3,3]^2:", cnt)

cnt2 = {True: 0, False: 0}
R2 = range(-2, 3)
cells = list(itertools.product(R2, R2))
for g1, g2 in itertools.combinations_with_replacement(cells, 2):
    for t in cells:
        cnt2[check_all([list(g1), list(g2)], list(t))] += 1
print("hai ma, đích trong [-2,2]^2:", cnt2)
assert cnt[True] > 100 and cnt[False] > 100 and cnt2[True] > 100 and cnt2[False] > 100

một ma trong [-3,3]^2, đích trong [-3,3]^2: {True: 1464, False: 937}


hai ma, đích trong [-2,2]^2: {True: 2948, False: 5177}


In [19]:
# 4) Ngẫu nhiên có seed, oracle BFS
rng = random.Random(789)
res = {True: 0, False: 0}
for _ in range(1500):
    g = rng.randint(1, 4)
    ghosts = [[rng.randint(-6, 6), rng.randint(-6, 6)] for _ in range(g)]
    target = [rng.randint(-6, 6), rng.randint(-6, 6)]
    res[check_all(ghosts, target)] += 1
print("Ngẫu nhiên nhỏ:", res)
assert res[True] > 100 and res[False] > 100

# 5) Tọa độ lớn: 100 ma, so Cách 2, 3, 4
rng = random.Random(10**4)
res = {True: 0, False: 0}
for k in range(30):
    n = rng.choice([1, 5, 100])
    target = [rng.randint(-10**4, 10**4), rng.randint(-10**4, 10**4)]
    if k % 3 == 0:                                       # ma cách đích gần bằng khoảng cách của bạn
        d = abs(target[0]) + abs(target[1])
        ghosts = []
        for _ in range(n):
            gd = d + rng.choice([-1, 0, 1, 2, 5] if k % 6 == 0 else [1, 2, 5])        # sát nút (nhóm sau: mọi ma xa hơn)
            dx = rng.randint(0, min(gd, 10**4))
            sx = rng.choice([-1, 1]); sy = rng.choice([-1, 1])
            gx, gy = target[0] + sx * dx, target[1] + sy * max(0, gd - dx)
            ghosts.append([max(-10**4, min(10**4, gx)), max(-10**4, min(10**4, gy))])
    else:
        ghosts = [[rng.randint(-10**4, 10**4), rng.randint(-10**4, 10**4)] for _ in range(n)]
    res[check_all(ghosts, target, with_bfs=False)] += 1
print("Tọa độ lớn (Cách 2, 3, 4 trùng nhau):", res)
assert res[True] > 3 and res[False] > 3

Ngẫu nhiên nhỏ: {True: 605, False: 895}


Tọa độ lớn (Cách 2, 3, 4 trùng nhau): {True: 9, False: 21}


In [20]:
# 6) Tính chất
rng = random.Random(6)
for _ in range(2000):
    g = rng.randint(1, 4)
    ghosts = [[rng.randint(-6, 6), rng.randint(-6, 6)] for _ in range(g)]
    target = [rng.randint(-6, 6), rng.randint(-6, 6)]
    base = escape_formula(ghosts, target)
    extra = [rng.randint(-6, 6), rng.randint(-6, 6)]
    more = escape_formula(ghosts + [extra], target)
    assert (not base) <= (not more)                           # thêm ma: không thể từ False sang True
    assert not escape_formula(ghosts + [list(target)], target)  # ma trên đích: luôn False
    # phản xạ qua trục x và đảo trục: khoảng cách Manhattan bất biến
    flip = [[-x, y] for x, y in ghosts], [-target[0], target[1]]
    swap = [[y, x] for x, y in ghosts], [target[1], target[0]]
    assert escape_formula(*flip) == base == escape_formula(*swap)
print("Tính chất (đơn điệu theo số ma, ma trên đích, đối xứng): OK")

# 7) Thời gian
big_ghosts = [[-10**4, 10**4]] * 100
big_target = [10**4, 10**4 - 1]
for name in ("formula", "path", "sim"):
    t0 = time.perf_counter()
    res_ = IMPLS[name](big_ghosts, big_target)
    print(f"{name:<8} g=100, D≈2·10^4: {(time.perf_counter() - t0) * 1000:9.3f} ms  -> {res_}")
t0 = time.perf_counter(); escape_bfs([[4, 4]], [3, 1]); print(f"bfs      tọa độ nhỏ ~4:       {(time.perf_counter() - t0) * 1000:9.3f} ms")
t0 = time.perf_counter(); escape_bfs([[14, 14]], [13, 11]); print(f"bfs      tọa độ ~14:          {(time.perf_counter() - t0) * 1000:9.3f} ms (tăng rất nhanh)")

Tính chất (đơn điệu theo số ma, ma trên đích, đối xứng): OK
formula  g=100, D≈2·10^4:     0.015 ms  -> True


path     g=100, D≈2·10^4:   232.604 ms  -> True


sim      g=100, D≈2·10^4:   331.117 ms  -> True
bfs      tọa độ nhỏ ~4:           0.700 ms
bfs      tọa độ ~14:              7.480 ms (tăng rất nhanh)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `>=` thay cho `>` (coi hòa là thoát)

"Cùng lúc" **không** tính là thoát; ma thắng khi `d_ma <= d_bạn`. Viết `>=` làm Ví dụ 3 sai.

In [21]:
def wrong_tie(ghosts, target):
    tx, ty = target
    d = abs(tx) + abs(ty)
    return all(abs(x - tx) + abs(y - ty) >= d for x, y in ghosts)      # sai: >=


print("Ví dụ 3 sai :", wrong_tie([[2, 0]], [1, 0]), "| đúng:", escape_formula([[2, 0]], [1, 0]))
assert wrong_tie([[2, 0]], [1, 0]) is True and escape_formula([[2, 0]], [1, 0]) is False

Ví dụ 3 sai : True | đúng: False


### Lỗi 2: đo khoảng cách **ma → bạn** (gốc) thay vì **ma → đích**

Ma không cần đuổi bạn; nó chỉ cần tới **đích**. So `dist(ma, gốc)` với `dist(bạn, đích)` là so hai thứ không liên quan.

In [22]:
def wrong_ghost_to_me(ghosts, target):
    d = abs(target[0]) + abs(target[1])
    return all(abs(x) + abs(y) > d for x, y in ghosts)                   # sai: khoảng cách ma-gốc


# ma ở (0,3): cách gốc 3 > d = 2 (so sai), nhưng cách đích (0,2) chỉ 1 < 2 nên bạn thua
ghosts, target = [[0, 3]], [0, 2]
print("sai :", wrong_ghost_to_me(ghosts, target), "| đúng:", escape_formula(ghosts, target), "(ma cách đích 1 < 2)")
assert wrong_ghost_to_me(ghosts, target) is True and escape_formula(ghosts, target) is False
assert escape_bfs(ghosts, target) is False                  # oracle xác nhận

sai : True | đúng: False (ma cách đích 1 < 2)


### Lỗi 3: `return True` sớm trong vòng lặp qua các ma

Chỉ cần **một** ma nguy hiểm là thua; trả `True` ngay khi gặp một ma "an toàn" bỏ qua các ma sau.

In [23]:
def wrong_early_true(ghosts, target):
    tx, ty = target
    d = abs(tx) + abs(ty)
    for x, y in ghosts:
        if abs(x - tx) + abs(y - ty) > d:
            return True                                                  # sai: còn các ma khác
    return False


ghosts, target = [[100, 100], [1, 0]], [2, 0]
print("sai :", wrong_early_true(ghosts, target), "| đúng:", escape_formula(ghosts, target))
assert wrong_early_true(ghosts, target) is True and escape_formula(ghosts, target) is False

sai : True | đúng: False


### Lỗi 4: dùng khoảng cách **Euclid** (hoặc Chebyshev) thay vì Manhattan

Mỗi lượt chỉ đi 4 hướng (không đi chéo), nên đường đi tối thiểu là Manhattan. Euclid/Chebyshev đánh giá thấp thời gian.

In [24]:
def wrong_euclid(ghosts, target):
    tx, ty = target
    d = (tx * tx + ty * ty) ** 0.5
    return all(((x - tx) ** 2 + (y - ty) ** 2) ** 0.5 > d for x, y in ghosts)


# tìm các ca mà Euclid cho đáp án khác Manhattan (và oracle xác nhận Manhattan đúng)
cands = []
rng = random.Random(4)
for _ in range(3000):
    g = [[rng.randint(-6, 6), rng.randint(-6, 6)]]
    t = [rng.randint(-6, 6), rng.randint(-6, 6)]
    if wrong_euclid(g, t) != escape_formula(g, t):
        cands.append((g, t))
print("số ca ngẫu nhiên mà Euclid cho đáp án sai:", len(cands), "| ví dụ:", cands[0])
assert cands and all(escape_bfs(g, t) == escape_formula(g, t) != wrong_euclid(g, t) for g, t in cands[:50])

số ca ngẫu nhiên mà Euclid cho đáp án sai: 176 | ví dụ: ([[-3, -2]], [-5, 5])


### Lỗi 5: chỉ xét các ma "nằm giữa đường" đi tới đích

Trực giác "ma phải chặn đường" sai: ma ở **bên cạnh** hoặc **phía sau** đích vẫn có thể tới đích cùng lúc hoặc sớm hơn rồi đứng đợi (xem bẫy `[[1,1]]`, đích `[1,0]` ở mục 7).

In [25]:
def wrong_between_only(ghosts, target):
    tx, ty = target
    for gx, gy in ghosts:
        between = min(0, tx) <= gx <= max(0, tx) and min(0, ty) <= gy <= max(0, ty)   # ma trong hình chữ nhật gốc-đích
        if between and abs(gx - tx) + abs(gy - ty) <= abs(tx) + abs(ty):
            return False
    return True


cases = []
rng = random.Random(5)
for _ in range(3000):
    g = [[rng.randint(-6, 6), rng.randint(-6, 6)]]
    t = [rng.randint(-6, 6), rng.randint(-6, 6)]
    if wrong_between_only(g, t) != escape_formula(g, t):
        cases.append((g, t))
g, t = cases[0]
print("ví dụ lỗi:", g, t, "sai:", wrong_between_only(g, t), "| đúng:", escape_formula(g, t))
assert escape_bfs(g, t) == escape_formula(g, t) != wrong_between_only(g, t)

ví dụ lỗi: [[3, -2]] [5, -1] sai: True | đúng: False


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** trò chơi "ai tới trước" với mọi bên đi cùng tốc độ trên lưới ⇒ thường **thu về so sánh khoảng cách** tới mục tiêu chung; đối thủ tối ưu **lao thẳng tới mục tiêu**, không cần đuổi bạn.
- **Hỏi: "đối thủ tốt nhất làm gì?"** rồi chứng minh bằng bất đẳng thức tam giác thay vì mô phỏng.
- **Hòa = thua:** đọc kỹ điều kiện "cùng lúc không tính"; thử với số nhỏ để chọn `<` hay `<=`.
- **Khoảng cách Manhattan** cho di chuyển 4 hướng; **Chebyshev** (`max(|dx|, |dy|)`) nếu được đi chéo; Euclid hầu như không bao giờ đúng trên lưới.
- **Kiểm chứng công thức bằng oracle:** viết một mô phỏng ngây thơ (BFS theo lớp thời gian) cho input nhỏ rồi vét cạn — cách tốt nhất để tin một công thức "quá đẹp".
- **Cùng họ:** *Cat and Mouse*, *Escape a Large Maze* (1036), *Minimum Time Visiting All Points* (1266), *Walking Robot Simulation*, các bài "ai đến trước" trong lý thuyết trò chơi trên đồ thị.

## 15. Biến thể của bài

1. **Cho phép đi chéo (8 hướng):** thay Manhattan bằng khoảng cách **Chebyshev** `max(|dx|, |dy|)`.
2. **Ma đi nhanh gấp `k` lần:** ma tới đích khi `dist / k` (làm tròn lên) không lớn hơn thời gian của bạn.
3. **Bạn xuất phát từ điểm bất kỳ** (không phải gốc): thay `(0,0)` bằng `start`.
4. **Có chướng ngại vật:** khoảng cách phải tính bằng **BFS** (đa nguồn), không còn là Manhattan.
5. **Đếm số ma có thể chặn bạn** hoặc **số lượt tối đa bạn có thể chờ** trước khi đi (bài tập 5).
6. **Ma phải đi (không được đứng yên):** mô hình thay đổi tính chẵn lẻ (parity) — cần xét chẵn/lẻ của khoảng cách.

Cài nhanh biến thể 1 (đi chéo) và kiểm tra bằng oracle BFS (tham số `moves`, `metric`):

In [26]:
MOVES8 = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1)]          # gồm đứng yên + 8 hướng


def cheb(p, q):
    return max(abs(p[0] - q[0]), abs(p[1] - q[1]))


def escape_chebyshev(ghosts, target):
    tx, ty = target
    d = max(abs(tx), abs(ty))
    return all(max(abs(x - tx), abs(y - ty)) > d for x, y in ghosts)


rng = random.Random(15)
res = {True: 0, False: 0}
for _ in range(1500):
    g = rng.randint(1, 4)
    ghosts = [[rng.randint(-6, 6), rng.randint(-6, 6)] for _ in range(g)]
    target = [rng.randint(-6, 6), rng.randint(-6, 6)]
    want = escape_bfs(ghosts, target, moves=MOVES8, metric=cheb)
    assert escape_chebyshev(ghosts, target) == want, (ghosts, target)
    res[want] += 1
print("Biến thể đi chéo (Chebyshev) khớp oracle BFS 8 hướng trên 1500 test:", res)

Biến thể đi chéo (Chebyshev) khớp oracle BFS 8 hướng trên 1500 test: {True: 636, False: 864}


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS theo lớp thời gian | `S_t` các ô an toàn, loại ô trong "quả bóng" ma | `O(T·W·g)` | `O(W)` | oracle trên input nhỏ; biến thể không có công thức |
| 2. Công thức khoảng cách | `D < dist(ma, đích)` với mọi ma | `O(g)` | `O(1)` | **làm bài** |
| 3. Kiểm tra từng ô trên đường thẳng | `dist(ma, P_t) ≤ t` ở mọi ô | `O(D·g)` | `O(D)` | khi chưa tin công thức, muốn kiểm trực tiếp |
| 4. Mô phỏng song song | mọi ma lao thẳng tới đích | `O(D·g)` | `O(g)` | minh họa chiến lược của ma, dễ mở rộng sang tốc độ khác nhau |

Trực giác chọn: công thức (Cách 2) là mặc định; Cách 1 là "giám khảo" kiểm tra công thức; Cách 3, 4 là hai cách **nhìn** để hiểu vì sao công thức đúng.

## 17. 5 bài tập vận dụng (dễ → khó)

Mọi cell TODO **không gọi** hàm khi chạy notebook; đáp án có assert ở mục 19.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

```python
print(escape_formula([[3, 0]], [2, 0]))
print(escape_formula([[3, 0]], [-2, 0]))
print(escape_formula([[1, 1], [5, 5]], [1, 0]))
```

In [27]:
print(escape_formula([[3, 0]], [2, 0]))
print(escape_formula([[3, 0]], [-2, 0]))
print(escape_formula([[1, 1], [5, 5]], [1, 0]))

False
True
False


### Bài 2 (tính tay)

`ghosts = [[-2, 3], [4, 4]]`, `target = [3, 2]`. Tính `d` (bạn), khoảng cách của từng ma tới đích và kết luận. Ghi vào `my_answer_2` (kiểu `bool`; để `None` đến khi tính xong).

In [28]:
ghosts_b2, target_b2 = [[-2, 3], [4, 4]], [3, 2]
my_answer_2 = None          # TODO: True hoặc False sau khi tính tay

### Bài 3 (điền TODO)

Hoàn thành hàm `escape_todo` theo Cách 2. Cell chỉ **định nghĩa**, không gọi.

In [29]:
def escape_todo(ghosts, target):
    tx, ty = target
    d = abs(tx) + abs(ty)
    # TODO: trả về True nếu MỌI ma đều cách đích XA HƠN d (so sánh chặt)
    pass

### Bài 4 (tự viết lại)

Tự viết `escape_mine(ghosts, target)` theo **Cách 3** (kiểm tra từng ô trên đường đi thẳng: ô ở lượt `t` bị ma chặn nếu `dist(ma, ô) <= t`). Không nhìn code cũ. Cell dưới chỉ để khung rỗng.

In [30]:
def escape_mine(ghosts, target):
    # TODO: viết theo Cách 3
    pass

### Bài 5 (biến thể / mini challenge)

Viết `max_wait(ghosts, target)`: nếu bạn **không** thể thoát (kể cả không chờ) trả `-1`; ngược lại trả **số lượt tối đa** bạn có thể **đứng yên ở gốc** trước khi đi mà vẫn thoát. Gợi ý: đứng yên `w` lượt rồi đi thẳng tương đương việc bạn cần `D + w` lượt.

In [31]:
def max_wait(ghosts, target):
    # TODO: trả về -1 hoặc số lượt chờ tối đa
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: tính `d` của bạn rồi khoảng cách từng ma tới đích.
- Bài 2: `d = |3| + |2|`.
- Bài 3: `all(... > d for x, y in ghosts)`.
- Bài 4: dựng danh sách các ô của đường đi (theo `x` rồi `y`), `enumerate` cho lượt `t`.
- Bài 5: bạn thoát khi `D + w < min khoảng cách ma → đích`.

**Mức 2 (vừa):**
- Bài 1: dòng 1: bạn cần 2 lượt; ma `(3,0)` cách `(2,0)` là 1 → ?
- Bài 2: ma `(-2,3)` cách `(3,2)`: `5 + 1`; ma `(4,4)` cách: `1 + 2`.
- Bài 3: `return all(abs(x - tx) + abs(y - ty) > d for x, y in ghosts)`.
- Bài 4: `if any(abs(gx - px) + abs(gy - py) <= t ...): return False`.
- Bài 5: `m = min(dist(g, target) for g in ghosts)`; số chờ tối đa `m - D - 1`.

**Mức 3 (gần đáp án):**
- Bài 1: `False`, `True`, `False`.
- Bài 2: `d = 5`; ma 1: 6; ma 2: 3 ≤ 5 ⇒ `False`.
- Bài 3: đúng một dòng như mức 2.
- Bài 4: đường đi gồm `D + 1` ô, lượt `t` ứng với ô thứ `t`.
- Bài 5: `return m - D - 1 if m > D else -1`.

## 19. Đáp án tham khảo

Phần này đặt cuối bài giảng; mọi đáp án có assert.

In [32]:
# Bài 1
assert [escape_formula([[3, 0]], [2, 0]), escape_formula([[3, 0]], [-2, 0]),
        escape_formula([[1, 1], [5, 5]], [1, 0])] == [False, True, False]
print("Bài 1: False True False")

# Bài 2
ans_2 = False
d_b2 = abs(target_b2[0]) + abs(target_b2[1])
dists = [dist(g, target_b2) for g in ghosts_b2]
assert d_b2 == 5 and dists == [6, 3]
assert escape_formula(ghosts_b2, target_b2) is ans_2 and escape_bfs(ghosts_b2, target_b2) is ans_2
print("Bài 2:", ans_2, f"(d = {d_b2}, khoảng cách ma = {dists}; ma thứ hai ≤ d)")


# Bài 3
def escape_todo_sol(ghosts, target):
    tx, ty = target
    d = abs(tx) + abs(ty)
    return all(abs(x - tx) + abs(y - ty) > d for x, y in ghosts)


rng = random.Random(3)
for _ in range(500):
    g = [[rng.randint(-6, 6), rng.randint(-6, 6)] for _ in range(rng.randint(1, 4))]
    t = [rng.randint(-6, 6), rng.randint(-6, 6)]
    assert escape_todo_sol(g, t) == escape_formula(g, t)
print("Bài 3: OK")

Bài 1: False True False
Bài 2: False (d = 5, khoảng cách ma = [6, 3]; ma thứ hai ≤ d)
Bài 3: OK


In [33]:
# Bài 4
def escape_mine_sol(ghosts, target):
    tx, ty = target
    pts = [(0, 0)]
    x = y = 0
    while x != tx:
        x += 1 if tx > x else -1
        pts.append((x, y))
    while y != ty:
        y += 1 if ty > y else -1
        pts.append((x, y))
    return not any(abs(gx - px) + abs(gy - py) <= t
                   for t, (px, py) in enumerate(pts) for gx, gy in ghosts)


rng = random.Random(4)
for _ in range(1000):
    g = [[rng.randint(-6, 6), rng.randint(-6, 6)] for _ in range(rng.randint(1, 4))]
    t = [rng.randint(-6, 6), rng.randint(-6, 6)]
    assert escape_mine_sol(g, t) == escape_formula(g, t) == escape_bfs(g, t)
print("Bài 4: OK")


# Bài 5
def max_wait_sol(ghosts, target):
    D = abs(target[0]) + abs(target[1])
    m = min(dist(g, target) for g in ghosts)
    return m - D - 1 if m > D else -1


rng = random.Random(5)
for _ in range(400):
    g = [[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(rng.randint(1, 3))]
    t = [rng.randint(-5, 5), rng.randint(-5, 5)]
    w = max_wait_sol(g, t)
    if w == -1:
        assert escape_bfs(g, t) is False
    else:
        assert escape_bfs(g, t, delay=w) is True and escape_bfs(g, t, delay=w + 1) is False
print("Bài 5: OK (đối chiếu oracle BFS có tham số delay)")

Bài 4: OK
Bài 5: OK (đối chiếu oracle BFS có tham số delay)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
class Solution:
    def escapeGhosts(self, ghosts, target):
        tx, ty = target
        d = abs(tx) + abs(ty)                                   # bạn cần d lượt
        return all(abs(x - tx) + abs(y - ty) > d for x, y in ghosts)   # mọi ma phải XA HƠN d (chặt)
```

**Công thức nhớ:** *bạn thoát ⇔ `dist(bạn, đích) < dist(mọi ma, đích)`; hòa là thua.*

**Checklist trước khi submit:**

- [ ] Khoảng cách là **Manhattan** (4 hướng), tính từ **ma tới đích** (không phải ma tới bạn).
- [ ] So sánh **chặt** `>` (cùng lúc không tính là thoát).
- [ ] Dùng `all(...)` trên **mọi** ma; không `return True` sớm trong vòng lặp.
- [ ] Đã thử: ma ở gốc, ma trên đích, ma phía sau, nhiều ma cùng vị trí, tọa độ `±10^4`, `100` ma.
- [ ] Đã kiểm chứng công thức bằng oracle BFS trên input nhỏ (vét cạn).